In [8]:
########################################### Step 2: Simulated missing values in y1  #########################################
import pandas as pd
import numpy as np
import time

# === Load your full, complete dataset ===
df = pd.read_csv(r"C:\Users\abuoliem\Box\2.Second year2024\2.Spring2025\1.Research\2.satellite image analysis of pre- and post-disaster images\K.final files\merged_x_prime_y_prime_with_features_150%.csv", header=None)
df.columns = ["x_prime", "y_prime", "y1", "y2", "y3", "y4", "y5", "y6", "y7"]

# === Function to simulate missingness ===
def simulate_missingness(df, target_col='y1', missing_rates=[0.1, 0.2, 0.3, 0.4, 0.5], num_trials=5, seed=123):
    """
    Randomly simulate missing values in the target column for different missing rates and multiple trials.
    
    Returns:
        results: a list of dictionaries with:
            - missing_rate (e.g., 0.1)
            - trial (e.g., 1, 2, 3, 4, 5)
            - df_missing: the DataFrame with missing y1 values
            - missing_mask: boolean array, True where y1 is missing
    """
    results = []
    total_rows = len(df)

    for rate in missing_rates:  # Loop over each missing rate
        num_missing = int(rate * total_rows)
        for trial in range(1, num_trials + 1):  # Repeat 5 times
            start_time = time.time()  # Start timing
            
            np.random.seed(seed + trial)  # Ensure different random pattern every trial

            # Make a fresh copy
            df_trial = df.copy()

            # Choose rows to make missing
            missing_indices = np.random.choice(df.index, size=num_missing, replace=False)
            missing_mask = np.zeros(total_rows, dtype=bool)
            missing_mask[missing_indices] = True

            # Apply missingness
            df_trial.loc[missing_indices, target_col] = np.nan
            
            end_time = time.time()  # End timing
            duration_sec = end_time - start_time

            # Save results
            results.append({
                'missing_rate': rate,
                'trial': trial,
                'df_missing': df_trial,
                'missing_mask': missing_mask,
                'duration_sec': duration_sec
            })
    
    return results

# === Run the function to generate all simulations ===
missing_data_trials = simulate_missingness(df, target_col='y1')

# === Print summary for each trial ===
print(f"Generated {len(missing_data_trials)} trials.")
print("Summary of all trials:\n")

for result in missing_data_trials:
    print(f"Missing Rate: {int(result['missing_rate'] * 100)}%, "
          f"Trial: {result['trial']}, "
          f"Missing Count: {result['df_missing']['y1'].isna().sum()}, "
          f"Time: {result['duration_sec']:.4f} seconds")


In [9]:
import pandas as pd
import numpy as np
import time
import os
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LeakyReLU, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# First define the model function
def build_best_model():
    model = Sequential()
    model.add(Input(shape=(6,)))
    act = LeakyReLU(negative_slope=0.1)  # updated
    model.add(Dense(128))
    model.add(act)
    model.add(Dropout(0.1))
    model.add(Dense(128))
    model.add(act)
    model.add(Dropout(0.1))
    model.add(Dense(32))
    model.add(act)
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=0.01), loss='mae', metrics=['mae'])
    return model

# Build model
model = build_best_model()

# Check layer structure
for i, layer in enumerate(model.layers):
    print(f"Layer {i}: {layer.name}, Type: {type(layer)}")

# Find the first Dense layer with weights
for i, layer in enumerate(model.layers):
    weights = layer.get_weights()
    if weights:
        print(f"\n🎯 First Dense Layer with weights is layer {i}: {layer.name}")
        weights, biases = weights
        print("Initial weights shape:", weights.shape)
        print("Initial biases shape:", biases.shape)
        print("Sample weights:\n", weights[:5])
        print("Sample biases:\n", biases[:5])
        break



In [10]:
########################################### Step 3: Train DL Model and predict  #########################################
import pandas as pd
import numpy as np
import time
import os
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LeakyReLU, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Assumes:
# - `df` is your original dataset (with y1 complete)
# - `missing_data_trials` is your list of 25 simulations from Step 1
# Define your custom folder path
results_dir = r"C:\Users\abuoliem\Box\2.Second year2024\2.Spring2025\1.Research\2.satellite image analysis of pre- and post-disaster images\L.DL\150%_model_Results"

# Create the folder if it doesn't exist
os.makedirs(results_dir, exist_ok=True)


# === Define the DL model (based on best grid search settings) ===
def build_best_model():
    model = Sequential()
    model.add(Input(shape=(6,)))
    act = LeakyReLU(alpha=0.1)
    model.add(Dense(128, activation=act))
    model.add(Dropout(0.1))
    model.add(Dense(128, activation=act))
    model.add(Dropout(0.1))
    model.add(Dense(32, activation=act))
    model.add(Dense(1))  # output layer
    model.compile(optimizer=Adam(learning_rate=0.01), loss='mae', metrics=['mae'])
    return model

# === List to store results from each trial ===
metrics = []

# === Loop over each simulated trial ===
for i, trial_data in enumerate(missing_data_trials, start=1):
    df_trial = trial_data['df_missing'].copy()
    missing_mask = trial_data['missing_mask']
    full_y1 = df['y1'].values  # true y1 values before masking

    # === Prepare feature and target arrays ===
    X = df_trial[["y2", "y3", "y4", "y5", "y6", "y7"]].values
    y = df_trial["y1"].values

    X_train = X[~missing_mask]
    y_train = y[~missing_mask]
    X_pred = X[missing_mask]
    y_true = full_y1[missing_mask]

    # === Normalize input features ===
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_pred_scaled = scaler.transform(X_pred)

    # === Build and train the model ===
    model = build_best_model()
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    start_time = time.time()
    history = model.fit(X_train_scaled, y_train,
                        validation_split=0.2,
                        epochs=100,
                        batch_size=16,
                        callbacks=[early_stop],
                        verbose=0)
    end_time = time.time()
    duration = end_time - start_time

    # === Predict missing values ===
    y_pred = model.predict(X_pred_scaled).flatten()

    # Residuals for this trial
    residuals = np.abs(y_pred - y_true)

    # === Compute evaluation metrics ===
    mae = mean_absolute_error(y_true, y_pred)
    se = np.std(np.abs(y_pred - y_true)) / np.sqrt(len(y_true))
    sd = np.std(residuals)  # ✅ Standard Deviation per trial

    # Save prediction results to CSV
    output_df = df_trial.loc[missing_mask, ["x_prime", "y_prime"]].copy()
    output_df["true_y1"] = y_true
    output_df["predicted_y1"] = y_pred
    filename = f"{int(trial_data['missing_rate']*100)}pct_trial{trial_data['trial']}_DL.csv"
    output_df.to_csv(os.path.join(results_dir, filename), index=False)
    
    # === Store results ===
    metrics.append({
        'missing_rate': trial_data['missing_rate'],
        'trial': trial_data['trial'],
        'mae': mae,
        'se': se,
        'sd': sd,
        'duration_sec': duration
    })

    # === Plot loss curves for this trial ===
    plt.figure(figsize=(6, 4))
    plt.plot(history.history['loss'], label='Training MAE')
    plt.plot(history.history['val_loss'], label='Validation MAE')
    plt.xlabel('Epoch')
    plt.ylabel('MAE')
    plt.title(f'Trial {trial_data["trial"]} - {int(trial_data["missing_rate"]*100)}% Missing')
    plt.legend()
    plt.tight_layout()
    plt.show()



In [12]:
########################################### Step 4: Summary statistics  #########################################

metrics_df = pd.DataFrame(metrics)

# Print all trial-wise results
print("\nTrial-wise results:")
print(metrics_df.to_string(index=False, float_format="%.5f"))

# Save trial-wise results to CSV
metrics_df.to_csv(os.path.join(results_dir, "DL_Trial_Results.csv"), index=False)

# Compute averages per missing rate
summary_df = metrics_df.groupby('missing_rate').agg(
    AME=('mae', 'mean'),
    SE=('mae', lambda x: np.std(x) / np.sqrt(len(x))),
    SD=('mae', 'std'),
    Time_Sec=('duration_sec', 'mean')
).reset_index()

print("\nSummary of DL model performance by missing rate:")
print(summary_df.to_string(index=False, float_format="%.5f"))

# Save summary to CSV
summary_df.to_csv(os.path.join(results_dir, "DL_Summary_Avg.csv"), index=False)

# Optional confirmation message
print(f"\nSaved trial-wise and summary results to: {results_dir}")
